# Triagem automática de imagens — YOLO26x e YOLOE-26m (v7)

Este notebook executa **somente a inferência**. O programa de Windows organiza e **move os arquivos originais** para `deteccao`, `duvida` e `sem_deteccoes`.

**1.** Em `Ambiente de execução > Alterar tipo de ambiente de execução`, selecione GPU T4 ou superior, se disponível.

**2.** Execute as células em sequência. Na célula 3 será solicitado seu token ngrok. Copie a URL e a X-API-Key exibidas para a interface do Windows.

**Privacidade:** as imagens temporárias são enviadas pelo túnel ngrok ao Colab. Não use imagens de clientes/empresas sem autorização.

**Nota:** o Colab encerra sessões e o ngrok pode mudar a URL. Reconfigure a interface se isso acontecer.

In [ ]:
%pip install -q "ultralytics>=8.4.0" "fastapi>=0.110" "uvicorn>=0.27" "python-multipart>=0.0.9" "pyngrok>=7"

In [ ]:
# API versionada. Não precisa instalar nada no Windows além dos requisitos do pacote.
"""Servidor FastAPI para o Colab. Iniciado pelo notebook incluso.
Somente transfere JPEGs temporarios; arquivos originais permanecem no Windows.
"""
from __future__ import annotations
import gc
import hmac
import json
import threading
import uuid
from collections import Counter

import cv2
import numpy as np
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile


MODEL_FILES = {'yolo26': 'yolo26x.pt', 'yoloe26': 'yoloe-26m-seg.pt'}
IMG_SIZES = {'yolo26': 960, 'yoloe26': 768}
MAX_IMAGES = 8
MAX_BYTES = 12 * 1024 * 1024


class Motor:
    def __init__(self, chave):
        self.chave = chave
        self.lock = threading.RLock()
        self.modelo = None
        self.tipo = None
        self.prompts = ()
        self.classes = {}
        self.config_id = ''
        import torch
        self.torch = torch
        self.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    def autenticar(self, api_key):
        if not hmac.compare_digest(api_key or '', self.chave):
            raise HTTPException(401, detail='X-API-Key incorreta')

    @staticmethod
    def validar_prompts(raw):
        try:
            prompts = json.loads(raw or '[]')
        except ValueError as exc:
            raise HTTPException(422, detail='prompts: lista JSON invalida') from exc
        if not isinstance(prompts, list) or not 1 <= len(prompts) <= 20:
            raise HTTPException(422, detail='YOLOE precisa de 1 a 20 prompts')
        if any(not isinstance(p, str) or not 1 <= len(p.strip()) <= 80 for p in prompts):
            raise HTTPException(422, detail='Prompt invalido')
        prompts = tuple(p.strip() for p in prompts)
        if len({p.casefold() for p in prompts}) != len(prompts):
            raise HTTPException(422, detail='Prompts repetidos')
        return prompts

    def carregar(self, tipo, prompts=()):
        if tipo not in MODEL_FILES:
            raise HTTPException(422, detail='Modelo deve ser yolo26 ou yoloe26')
        if self.modelo is not None and self.tipo == tipo and self.prompts == prompts:
            return
        from ultralytics import YOLO, YOLOE
        # O modelo mais pesado sai da VRAM antes de entrar o proximo.
        if self.tipo != tipo:
            self.modelo = None
            self.tipo = None
            self.classes = {}
            self.prompts = ()
            gc.collect()
            if self.device.startswith('cuda'):
                self.torch.cuda.empty_cache()
            print('[Modelo] Carregando', MODEL_FILES[tipo], flush=True)
            modelo = YOLO(MODEL_FILES[tipo]) if tipo == 'yolo26' else YOLOE(MODEL_FILES[tipo])
            modelo.to(self.device)
            self.modelo = modelo
            self.tipo = tipo
            if tipo == 'yolo26':
                self.classes = {int(k): str(v) for k,v in modelo.names.items()}
        if tipo == 'yoloe26':
            print('[YOLOE] Preparando classes: ', prompts, flush=True)
            self.modelo.set_classes(list(prompts))
            self.classes = {i: p for i,p in enumerate(prompts)}
            self.prompts = prompts
        self.config_id = uuid.uuid4().hex

    def ids(self, raw):
        if not raw:
            return [0]
        try:
            ids = list(dict.fromkeys(int(x.strip()) for x in raw.split(',') if x.strip()))
        except ValueError as exc:
            raise HTTPException(422, detail='classes precisa de IDs numericos') from exc
        if not ids or any(i not in self.classes for i in ids):
            raise HTTPException(422, detail='Classes nao encontradas no checkpoint')
        return ids

    def predizer(self, imagens, conf, ids):
        """Falha OOM: reduz lote ou tamanho, preservando alinhamento de resposta."""
        tamanho = IMG_SIZES[self.tipo]
        def recursivo(frames, imgsz):
            try:
                kwargs = dict(source=frames, imgsz=imgsz, device=self.device, conf=conf,
                              verbose=False, max_det=300)
                if self.device.startswith('cuda'):
                    kwargs['quantize'] = 16
                if self.tipo == 'yolo26':
                    kwargs['classes'] = ids
                results = self.modelo.predict(**kwargs)
                if len(results) != len(frames):
                    raise RuntimeError('Inferencia retornou tamanho inesperado')
                return list(zip(results, [imgsz]*len(results)))
            except Exception as exc:
                oom = self.device.startswith('cuda') and (
                    isinstance(exc, self.torch.cuda.OutOfMemoryError) or 'out of memory' in str(exc).lower())
                if not oom:
                    raise
                print('[GPU] Sem memoria, reduzindo lote/resolucao.', flush=True)
                gc.collect()
                self.torch.cuda.empty_cache()
                if len(frames) > 1:
                    meio = len(frames)//2
                    return recursivo(frames[:meio], imgsz) + recursivo(frames[meio:], imgsz)
                if imgsz > 640:
                    return recursivo(frames, 640)
                raise RuntimeError('GPU sem memoria. Troque por checkpoint menor.') from exc
        return recursivo(imagens, tamanho)


def criar_app(chave: str) -> FastAPI:
    motor = Motor(chave)
    app = FastAPI(title='Triagem YOLO26 YOLOE26 - v7', version='7.0')

    @app.get('/health')
    def health(x_api_key: str = Header(default='', alias='X-API-Key')):
        motor.autenticar(x_api_key)
        return dict(status='ok', version='7.0', max_images=MAX_IMAGES, device=motor.device,
                    model_type=motor.tipo, model=MODEL_FILES.get(motor.tipo))

    @app.get('/classes')
    def classes(x_api_key: str = Header(default='', alias='X-API-Key')):
        motor.autenticar(x_api_key)
        with motor.lock:
            try:
                motor.carregar('yolo26')
            except Exception as exc:
                raise HTTPException(503, detail=f'Falha carregando YOLO26: {exc}') from exc
            return {'model': MODEL_FILES['yolo26'], 'classes': [dict(id=k,name=v) for k,v in sorted(motor.classes.items())]}

    @app.post('/prepare')
    def prepare(model_type: str = Form(...), prompts: str = Form(default=''),
                x_api_key: str = Header(default='', alias='X-API-Key')):
        motor.autenticar(x_api_key)
        pp = motor.validar_prompts(prompts) if model_type == 'yoloe26' else ()
        with motor.lock:
            try:
                motor.carregar(model_type, pp)
            except HTTPException:
                raise
            except Exception as exc:
                raise HTTPException(503, detail=f'Falha preparando modelo: {exc}') from exc
            return dict(status='ok', model=MODEL_FILES[model_type], model_type=model_type,
                        device=motor.device, imgsz=IMG_SIZES[model_type],
                        config_id=motor.config_id,
                        classes=[dict(id=k,name=v) for k,v in sorted(motor.classes.items())])

    @app.post('/predict-batch')
    async def predict_batch(files: list[UploadFile] = File(...),
                            config_id: str = Form(...),
                            classes: str = Form(default=''),
                            conf: float = Form(default=0.12, ge=0.01, le=1.0),
                            x_api_key: str = Header(default='', alias='X-API-Key')):
        motor.autenticar(x_api_key)
        if not 1 <= len(files) <= MAX_IMAGES:
            raise HTTPException(400, detail=f'Envie 1 a {MAX_IMAGES} arquivos JPEG')
        saidas = [None] * len(files)
        validos, indices = [], []
        for i, arquivo in enumerate(files):
            try:
                if arquivo.content_type not in ('image/jpeg', 'image/png', 'application/octet-stream'):
                    raise ValueError('Formatos aceitos: JPEG/PNG')
                conteudo = await arquivo.read(MAX_BYTES+1)
                if len(conteudo) > MAX_BYTES:
                    raise ValueError('Imagem acima de 12 MiB')
                frame = cv2.imdecode(np.frombuffer(conteudo, np.uint8), cv2.IMREAD_COLOR)
                if frame is None:
                    raise ValueError('Imagem invalida/corrompida')
                validos.append(frame)
                indices.append(i)
            except Exception as exc:
                saidas[i] = dict(index=i, status='erro', error=str(exc))
            finally:
                await arquivo.close()
        with motor.lock:
            if not motor.config_id or motor.config_id != config_id:
                raise HTTPException(409, detail='Configuracao alterada ou sessão reiniciada; prepare novamente')
            ids = motor.ids(classes) if motor.tipo == 'yolo26' else list(motor.classes)
            if validos:
                for start in range(0, len(validos), 2):
                    subset, subset_idx = validos[start:start+2], indices[start:start+2]
                    try:
                        inferidos = motor.predizer(subset, conf, ids)
                        for original_index, (r, imgsz) in zip(subset_idx, inferidos):
                            dets = []
                            if r.boxes is not None and len(r.boxes):
                                cids = r.boxes.cls.detach().cpu().tolist()
                                probs = r.boxes.conf.detach().cpu().tolist()
                                coords = r.boxes.xyxy.detach().cpu().tolist()
                                for cid, prob, xyxy in zip(cids, probs, coords):
                                    label_id = int(cid)
                                    dets.append(dict(id=label_id,
                                                     name=motor.classes.get(label_id, str(label_id)),
                                                     confidence=round(float(prob), 6),
                                                     xyxy=[round(float(v),1) for v in xyxy]))
                            saidas[original_index] = dict(index=original_index, status='ok',
                                                           detections=dets, num_deteccoes=len(dets),
                                                           maior_confianca=max([d['confidence'] for d in dets], default=0),
                                                           imgsz_utilizado=imgsz)
                    except Exception as exc:
                        for i in subset_idx:
                            saidas[i] = dict(index=i, status='erro', error=f'Falha de inferencia: {exc}')
            return dict(model_type=motor.tipo, model=MODEL_FILES[motor.tipo], results=saidas)
    return app


In [ ]:
import getpass, secrets, threading, time
import uvicorn
from pyngrok import ngrok

NGROK_TOKEN = getpass.getpass("Cole seu ngrok authtoken (oculto): ").strip()
if not NGROK_TOKEN:
    raise ValueError("Necessário informar authtoken do ngrok")
API_KEY = secrets.token_urlsafe(32)
app = criar_app(API_KEY)
PORT = 8000
if "server" in globals() and server.started:
    server.should_exit = True
    time.sleep(2)
config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning", access_log=False)
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
for _ in range(60):
    if server.started:
        break
    time.sleep(.5)
if not server.started:
    raise RuntimeError("Falha iniciando FastAPI")
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(addr=PORT, proto="http")
print("\n===== PREENCHA ESTES CAMPOS NO WINDOWS =====")
print("URL HTTPS:", tunnel.public_url)
print("X-API-Key:", API_KEY)
print("===========================================")
print("Não publique nem compartilhe a chave.")

In [ ]:
import requests
r=requests.get(tunnel.public_url.rstrip("/")+"/health",headers={"X-API-Key":API_KEY,"ngrok-skip-browser-warning":"true"},timeout=30)
r.raise_for_status()
print(r.json())
print("API online. Agora abra INICIAR_WINDOWS.bat em seu computador.")

## Encerrar o servidor (opcional)

Quando finalizar, execute a célula abaixo caso queira interromper a API imediatamente.

In [ ]:
# Descomente depois de terminar:
# ngrok.disconnect(tunnel.public_url)
# server.should_exit = True